In [1]:
import pandas
import numpy
import sklearn
import mlflow
import fastapi
import uvicorn
import joblib

print("All packages imported successfully!")

All packages imported successfully!


In [2]:
import numpy as np
import pandas as pd


np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "temperature": np.random.normal(70, 10, n),
    "vibration": np.random.normal(0.5, 0.2, n),
    "pressure": np.random.normal(30, 5, n)
})

df["failure"] = (
    (df["temperature"] > 80) |
    (df["vibration"] > 0.7) |
    (df["pressure"] > 35)
).astype(int)

df.head()

,temperature,vibration,pressure,failure
0,74.967142,0.779871,26.624109,1
1,68.617357,0.684927,29.277407,0
2,76.476885,0.511926,26.037900,0
3,85.230299,0.370613,28.460192,1
4,67.658466,0.639645,20.531927,0


In [3]:
import mlflow

mlflow.set_tracking_uri("file:///C:/mlflow")
mlflow.set_experiment("predictive-maintenance")

C:\Users\Pradeep V\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///C:/mlflow/382732611558538676', creation_time=1776951473991, experiment_id='382732611558538676', last_update_time=1776951473991, lifecycle_stage='active', name='predictive-maintenance', tags={}, trace_location=None, workspace='default'>

In [4]:
df["failure"].value_counts()

failure
0    591
1    409
Name: count, dtype: int64

In [5]:
import mlflow

mlflow.set_tracking_uri("file:///C:/mlflow")
mlflow.set_experiment("predictive-maintenance")

<Experiment: artifact_location='file:///C:/mlflow/382732611558538676', creation_time=1776951473991, experiment_id='382732611558538676', last_update_time=1776951473991, lifecycle_stage='active', name='predictive-maintenance', tags={}, trace_location=None, workspace='default'>

In [6]:
mlflow.set_experiment("predictive-maintenance")

<Experiment: artifact_location='file:///C:/mlflow/382732611558538676', creation_time=1776951473991, experiment_id='382732611558538676', last_update_time=1776951473991, lifecycle_stage='active', name='predictive-maintenance', tags={}, trace_location=None, workspace='default'>

In [7]:
# recreate features and target
X = df.drop("failure", axis=1)
y = df["failure"]

print("X and y recreated")

X and y recreated


In [8]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

# ✅ Set tracking (important)
mlflow.set_tracking_uri("file:///C:/mlflow")
mlflow.set_experiment("predictive-maintenance")

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# try multiple experiments (better than single run)
for n in [50, 100, 200]:

    with mlflow.start_run(run_name=f"rf_{n}_trees"):

        # model
        model = RandomForestClassifier(n_estimators=n, random_state=42)

        # log parameters
        mlflow.log_param("n_estimators", n)
        mlflow.log_param("random_state", 42)

        # train
        model.fit(X_train, y_train)

        # predict
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)

        print(f"n_estimators={n} → Accuracy={acc}")

        # log metrics
        mlflow.log_metric("accuracy", acc)

        # log model
        mlflow.sklearn.log_model(model, "model")

# save best model locally (example: last one)
joblib.dump(model, "model.pkl")

2026/04/23 21:04:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:04:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=50 → Accuracy=0.995


2026/04/23 21:04:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:04:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=100 → Accuracy=0.995


2026/04/23 21:04:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 21:04:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=200 → Accuracy=0.995


['model.pkl']

In [9]:
mlflow.get_tracking_uri()

'file:///C:/mlflow'

In [ ]:
!python -m mlflow ui --backend-store-uri file:///C:/mlflow --port 5001

In [ ]:
python -m uvicorn app.main:app --reload